# 🚀 Prompt Engineering System Design - Master Notebook

This notebook systematically applies foundational and advanced prompt engineering techniques to design a production-ready system prompt.

**Before starting:** Complete `App_Concept.md` to define your AI app and test task.

## 📦 Setup & Configuration

In [ ]:
# Install required packages
!pip install openai --upgrade --quiet

In [ ]:
import os
import json
from openai import OpenAI
from typing import List, Dict
import time

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Configuration
MODEL = "gpt-3.5-turbo"  # or "gpt-4" for better results
TEMPERATURE = 0.7
MAX_TOKENS = 500

print("✅ Setup complete!")

## 🎯 Define Your Test Task

**IMPORTANT:** Use the SAME test task across ALL techniques for meaningful comparison.

In [ ]:
# Define your app concept and test task
# Copy from your App_Concept.md

APP_NAME = "[Your App Name]"
APP_DESCRIPTION = "[Brief description]"

# This is the task you'll test with ALL techniques
TEST_TASK = """
[Your specific test task here]

Example: Analyze this customer complaint and provide a professional response:
"I ordered a product 2 weeks ago and it still hasn't arrived. The tracking hasn't updated in days. This is unacceptable!"
"""

print(f"App: {APP_NAME}")
print(f"Test Task: {TEST_TASK[:100]}...")

## 🛠️ Helper Functions

In [ ]:
def call_llm(messages: List[Dict], temperature: float = TEMPERATURE, max_tokens: int = MAX_TOKENS):
    """Helper function to call OpenAI API."""
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

def print_response(technique_name: str, response: str):
    """Pretty print responses."""
    print(f"\n{'='*80}")
    print(f"🔹 {technique_name}")
    print(f"{'='*80}")
    print(response)
    print(f"{'='*80}\n")

def save_result(technique: str, prompt: str, response: str, notes: str = ""):
    """Save results for later analysis."""
    result = {
        "technique": technique,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "prompt": prompt,
        "response": response,
        "notes": notes
    }
    
    # Append to results file
    try:
        with open('results.json', 'r') as f:
            results = json.load(f)
    except FileNotFoundError:
        results = []
    
    results.append(result)
    
    with open('results.json', 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"💾 Saved result for: {technique}")

print("✅ Helper functions loaded!")

---
# 📚 PART 1: Foundational Techniques
---

## 1️⃣ Zero-Shot Prompting

**Goal:** Test baseline performance with minimal instruction.

In [ ]:
# Zero-Shot System Prompt
zero_shot_system = f"You are a helpful {APP_NAME}."

messages = [
    {"role": "system", "content": zero_shot_system},
    {"role": "user", "content": TEST_TASK}
]

response = call_llm(messages)
print_response("Zero-Shot", response)

# Save for comparison
save_result("zero-shot", zero_shot_system, response, "Baseline with minimal instruction")

**📝 Observations:**
- Clarity: [Rate 1-5]
- Accuracy: [Rate 1-5]
- Completeness: [Rate 1-5]
- Notes: [Your observations]

## 2️⃣ Few-Shot Learning

**Goal:** Provide examples to guide the model's behavior.

In [ ]:
# Few-Shot with Examples
few_shot_system = f"You are a helpful {APP_NAME}."

# Add 3-5 examples specific to your task
messages = [
    {"role": "system", "content": few_shot_system},
    
    # Example 1
    {"role": "user", "content": "[Example input 1]"},
    {"role": "assistant", "content": "[Example output 1]"},
    
    # Example 2
    {"role": "user", "content": "[Example input 2]"},
    {"role": "assistant", "content": "[Example output 2]"},
    
    # Example 3
    {"role": "user", "content": "[Example input 3]"},
    {"role": "assistant", "content": "[Example output 3]"},
    
    # Actual task
    {"role": "user", "content": TEST_TASK}
]

response = call_llm(messages)
print_response("Few-Shot", response)

save_result("few-shot", str(messages), response, "With 3 examples")

**📝 Observations:**
- Improvement over zero-shot: [Yes/No]
- What changed: [Describe]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 3️⃣ Chain-of-Thought (CoT)

**Goal:** Encourage step-by-step reasoning.

In [ ]:
# Chain-of-Thought System Prompt
cot_system = f"""
You are a helpful {APP_NAME}. 
When responding, think through the problem step by step before providing your final answer.
Show your reasoning process.
"""

messages = [
    {"role": "system", "content": cot_system},
    {"role": "user", "content": f"{TEST_TASK}\n\nLet's think step by step:"}
]

response = call_llm(messages)
print_response("Chain-of-Thought", response)

save_result("chain-of-thought", cot_system, response, "Step-by-step reasoning")

**📝 Observations:**
- Did reasoning improve quality? [Yes/No]
- Is the logic clear? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 4️⃣ Step-Back Prompting

**Goal:** Consider broader principles before diving into specifics.

In [ ]:
# Step-Back: First get principles
stepback_system = f"You are a helpful {APP_NAME}."

# Step 1: Get high-level principles
messages_principles = [
    {"role": "system", "content": stepback_system},
    {"role": "user", "content": f"What are the key principles or best practices for handling this type of situation: {TEST_TASK}"}
]

principles = call_llm(messages_principles)
print("📚 Principles identified:")
print(principles)

# Step 2: Apply principles to specific task
messages_apply = [
    {"role": "system", "content": stepback_system},
    {"role": "user", "content": f"Given these principles:\n{principles}\n\nNow apply them to: {TEST_TASK}"}
]

response = call_llm(messages_apply)
print_response("Step-Back", response)

save_result("step-back", stepback_system + "\n" + principles, response, "Used high-level principles first")

**📝 Observations:**
- Did abstraction help? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 5️⃣ Analogical Prompting

**Goal:** Use analogies to improve understanding.

In [ ]:
# Analogical Prompting
analogical_system = f"""
You are a helpful {APP_NAME}.
When solving problems, consider similar situations you've encountered and use them as analogies.
"""

messages = [
    {"role": "system", "content": analogical_system},
    {"role": "user", "content": f"This situation is like [provide an analogy relevant to your domain].\n\nNow handle: {TEST_TASK}"}
]

response = call_llm(messages)
print_response("Analogical", response)

save_result("analogical", analogical_system, response, "Used domain analogy")

**📝 Observations:**
- Did analogies clarify the approach? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 6️⃣ Auto-CoT (Automatic Chain of Thought)

**Goal:** Let the model generate its own reasoning examples.

In [ ]:
# Step 1: Generate examples with reasoning
auto_cot_system = f"You are a helpful {APP_NAME}."

messages_generate = [
    {"role": "system", "content": auto_cot_system},
    {"role": "user", "content": f"Generate 3 diverse examples for a {APP_NAME} with step-by-step reasoning for each."}
]

generated_examples = call_llm(messages_generate, max_tokens=800)
print("🤖 Auto-generated examples:")
print(generated_examples)

# Step 2: Use generated examples
messages_task = [
    {"role": "system", "content": auto_cot_system},
    {"role": "user", "content": f"Here are examples with reasoning:\n{generated_examples}\n\nNow apply the same reasoning approach to: {TEST_TASK}"}
]

response = call_llm(messages_task)
print_response("Auto-CoT", response)

save_result("auto-cot", auto_cot_system + "\n" + generated_examples, response, "Model-generated reasoning examples")

**📝 Observations:**
- Quality of auto-generated examples: [Good/Fair/Poor]
- Vs. human-written examples: [Better/Worse/Same]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 7️⃣ Generate-Knowledge Prompting

**Goal:** Generate relevant knowledge before answering.

In [ ]:
# Step 1: Generate knowledge
gk_system = f"You are a helpful {APP_NAME}."

messages_knowledge = [
    {"role": "system", "content": gk_system},
    {"role": "user", "content": f"What knowledge, facts, or background information would be helpful to answer this: {TEST_TASK}"}
]

knowledge = call_llm(messages_knowledge)
print("📖 Generated knowledge:")
print(knowledge)

# Step 2: Use knowledge to answer
messages_answer = [
    {"role": "system", "content": gk_system},
    {"role": "user", "content": f"Using this knowledge:\n{knowledge}\n\nNow respond to: {TEST_TASK}"}
]

response = call_llm(messages_answer)
print_response("Generate-Knowledge", response)

save_result("generate-knowledge", gk_system + "\n" + knowledge, response, "Generated background knowledge first")

**📝 Observations:**
- Did knowledge generation improve accuracy? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

---
# 🚀 PART 2: Advanced Techniques
---

## 8️⃣ Decomposition

**Goal:** Break complex tasks into sub-tasks.

In [ ]:
# Identify sub-tasks
decomp_system = f"You are a helpful {APP_NAME}."

# Step 1: Decompose
messages_decompose = [
    {"role": "system", "content": decomp_system},
    {"role": "user", "content": f"Break this task into 3-5 sub-tasks: {TEST_TASK}"}
]

subtasks = call_llm(messages_decompose)
print("📋 Sub-tasks identified:")
print(subtasks)

# Step 2: Execute each sub-task (simplified - you can make this more sophisticated)
messages_execute = [
    {"role": "system", "content": decomp_system},
    {"role": "user", "content": f"Sub-tasks:\n{subtasks}\n\nNow execute each sub-task for: {TEST_TASK}"}
]

response = call_llm(messages_execute, max_tokens=800)
print_response("Decomposition", response)

save_result("decomposition", decomp_system + "\n" + subtasks, response, "Broke into sub-tasks")

**📝 Observations:**
- Did decomposition improve completeness? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 9️⃣ Ensembling

**Goal:** Generate multiple responses and synthesize the best elements.

In [ ]:
# Generate multiple responses with different temperatures
ensemble_system = f"You are a helpful {APP_NAME}."

messages = [
    {"role": "system", "content": ensemble_system},
    {"role": "user", "content": TEST_TASK}
]

responses = []
temperatures = [0.3, 0.7, 1.0]

print("Generating multiple responses...\n")
for i, temp in enumerate(temperatures, 1):
    response = call_llm(messages, temperature=temp)
    responses.append(response)
    print(f"Response {i} (temp={temp}):")
    print(response[:200] + "...\n")

# Synthesize
synthesis_prompt = f"""
I have {len(responses)} different responses to the same task. 
Synthesize the best elements from all responses into one high-quality response.

Response 1: {responses[0]}

Response 2: {responses[1]}

Response 3: {responses[2]}

Synthesized response:
"""

messages_synth = [
    {"role": "system", "content": ensemble_system},
    {"role": "user", "content": synthesis_prompt}
]

synthesized = call_llm(messages_synth, max_tokens=800)
print_response("Ensembling (Synthesized)", synthesized)

save_result("ensembling", ensemble_system, synthesized, f"Synthesized from {len(responses)} responses")

**📝 Observations:**
- Did synthesizing improve quality? [Yes/No]
- Worth the extra cost? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 🔟 Self-Consistency

**Goal:** Generate multiple reasoning paths, pick most consistent answer.

In [ ]:
# Generate 5 responses with CoT
sc_system = f"""
You are a helpful {APP_NAME}.
Provide step-by-step reasoning for your answer.
"""

messages = [
    {"role": "system", "content": sc_system},
    {"role": "user", "content": f"{TEST_TASK}\n\nLet's think step by step:"}
]

reasoning_paths = []
print("Generating multiple reasoning paths...\n")

for i in range(5):
    response = call_llm(messages, temperature=0.8)
    reasoning_paths.append(response)
    print(f"Path {i+1}: {response[:150]}...\n")

# Find most consistent answer
consistency_prompt = f"""
I have 5 different reasoning paths for the same task.
Analyze them and identify the most consistent/common conclusion.
Then provide that conclusion with the best reasoning.

Path 1: {reasoning_paths[0]}

Path 2: {reasoning_paths[1]}

Path 3: {reasoning_paths[2]}

Path 4: {reasoning_paths[3]}

Path 5: {reasoning_paths[4]}

Most consistent answer:
"""

messages_consistent = [
    {"role": "system", "content": sc_system},
    {"role": "user", "content": consistency_prompt}
]

consistent_answer = call_llm(messages_consistent, max_tokens=800)
print_response("Self-Consistency", consistent_answer)

save_result("self-consistency", sc_system, consistent_answer, "Most consistent from 5 reasoning paths")

**📝 Observations:**
- Did consistency improve reliability? [Yes/No]
- How much variance in paths? [High/Medium/Low]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 1️⃣1️⃣ Universal Self-Consistency

**Goal:** Apply self-consistency across different prompt formulations.

In [ ]:
# Create 3 different prompt formulations
usc_prompts = [
    f"You are a helpful {APP_NAME}. Be thorough and systematic.",
    f"You are an expert {APP_NAME}. Provide actionable advice.",
    f"You are a professional {APP_NAME}. Focus on quality and accuracy."
]

all_responses = []

print("Generating responses across different formulations...\n")

for i, system_prompt in enumerate(usc_prompts, 1):
    print(f"Formulation {i}:")
    for j in range(3):  # 3 responses per formulation
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": TEST_TASK}
        ]
        response = call_llm(messages, temperature=0.8)
        all_responses.append(response)
        print(f"  Response {j+1}: {response[:100]}...")
    print()

# Aggregate across all
aggregation_prompt = f"""
I have {len(all_responses)} responses from different prompt formulations.
Find the common themes and synthesize the most reliable answer.

""" + "\n\n".join([f"Response {i+1}: {r}" for i, r in enumerate(all_responses)])

messages_agg = [
    {"role": "system", "content": "You are a helpful analyst."},
    {"role": "user", "content": aggregation_prompt}
]

universal_answer = call_llm(messages_agg, max_tokens=800)
print_response("Universal Self-Consistency", universal_answer)

save_result("universal-self-consistency", "Multiple formulations", universal_answer, f"Aggregated from {len(all_responses)} responses")

**📝 Observations:**
- Most robust technique so far? [Yes/No]
- Worth the complexity? [Yes/No]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

## 1️⃣2️⃣ Self-Criticism

**Goal:** Have model critique and improve its own output.

In [ ]:
# Initial response
sc_system = f"You are a helpful {APP_NAME}."

messages_initial = [
    {"role": "system", "content": sc_system},
    {"role": "user", "content": TEST_TASK}
]

initial_response = call_llm(messages_initial)
print("Initial response:")
print(initial_response)
print("\n" + "="*80 + "\n")

# Iteration 1: Critique
messages_critique = [
    {"role": "system", "content": sc_system},
    {"role": "user", "content": f"Critique this response. What could be improved?\n\n{initial_response}"}
]

critique = call_llm(messages_critique)
print("Critique:")
print(critique)
print("\n" + "="*80 + "\n")

# Iteration 2: Improve
messages_improve = [
    {"role": "system", "content": sc_system},
    {"role": "user", "content": f"Based on this critique: {critique}\n\nImprove this response: {initial_response}"}
]

improved_v1 = call_llm(messages_improve)
print("Improved Version 1:")
print(improved_v1)
print("\n" + "="*80 + "\n")

# Iteration 3: Final refinement
messages_final = [
    {"role": "system", "content": sc_system},
    {"role": "user", "content": f"Make final improvements to ensure this is production-ready: {improved_v1}"}
]

final_response = call_llm(messages_final)
print_response("Self-Criticism (Final)", final_response)

save_result("self-criticism", sc_system, final_response, "After 3 iterations of self-critique")

**📝 Observations:**
- How much did quality improve? [Significant/Moderate/Minimal]
- Number of iterations needed: [__]
- Quality ratings: Clarity[__] Accuracy[__] Completeness[__]

---
# 📊 PART 3: Evaluation & Comparison
---

## Load and Compare All Results

In [ ]:
# Load all saved results
with open('results.json', 'r') as f:
    all_results = json.load(f)

print(f"Total techniques tested: {len(all_results)}\n")
print("Techniques:")
for result in all_results:
    print(f"  - {result['technique']}")

## Create Comparison Matrix

Rate each technique on a 1-5 scale:

In [ ]:
# Create comparison data
# Fill in your ratings after reviewing all responses

comparison = {
    "zero-shot": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 1},
    "few-shot": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 3},
    "chain-of-thought": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 2},
    "step-back": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 3},
    "analogical": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 2},
    "auto-cot": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 2},
    "generate-knowledge": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 3},
    "decomposition": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 4},
    "ensembling": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 5},
    "self-consistency": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 5},
    "universal-self-consistency": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 5},
    "self-criticism": {"clarity": 0, "accuracy": 0, "completeness": 0, "consistency": 0, "effort": 4},
}

# Display as table
import pandas as pd

df = pd.DataFrame(comparison).T
df['total_quality'] = df[['clarity', 'accuracy', 'completeness', 'consistency']].sum(axis=1)
df = df.sort_values('total_quality', ascending=False)

print("\n📊 Technique Comparison Matrix:")
print(df)

---
# 🎯 PART 4: Final System Prompt Design
---

## Design Your Production System Prompt

Based on all experiments, design your final system prompt:

In [ ]:
# Your final, production-ready system prompt
# Incorporate the best techniques you discovered

FINAL_SYSTEM_PROMPT = """
[Your refined system prompt here - incorporate the techniques that worked best]

Example structure:
- Role definition
- Key principles (from step-back)
- Reasoning approach (from CoT)
- Examples (from few-shot)
- Quality guidelines
- Format instructions
"""

print("🎯 Final System Prompt:")
print("="*80)
print(FINAL_SYSTEM_PROMPT)
print("="*80)

# Save to file
with open('final_system_prompt.txt', 'w') as f:
    f.write(FINAL_SYSTEM_PROMPT)

print("\n💾 Saved to: final_system_prompt.txt")

## Test Final System Prompt

In [ ]:
# Test with your original task
messages = [
    {"role": "system", "content": FINAL_SYSTEM_PROMPT},
    {"role": "user", "content": TEST_TASK}
]

final_test = call_llm(messages, max_tokens=800)
print_response("Final System Prompt Test", final_test)

# Test with 2-3 additional test cases
test_cases = [
    "[Additional test case 1]",
    "[Additional test case 2]",
    "[Additional test case 3 - edge case]"
]

for i, test_case in enumerate(test_cases, 1):
    messages = [
        {"role": "system", "content": FINAL_SYSTEM_PROMPT},
        {"role": "user", "content": test_case}
    ]
    response = call_llm(messages, max_tokens=800)
    print_response(f"Test Case {i}", response)

---
# 📝 PART 5: Reflection
---

## Critical Reflection Questions

Answer these thoroughly for your submission:

### 1. Most Effective Technique
**Which technique produced the highest quality output for your specific task?**

[Your answer]

### 2. Ease of Implementation
**Which technique was easiest to implement and gave good ROI?**

[Your answer]

### 3. Effort vs. Benefit
**Which techniques required too much effort for minimal benefit?**

[Your answer]

### 4. Impact of Examples (Few-Shot)
**Did providing examples significantly improve results? Why or why not?**

[Your answer]

### 5. Impact of Reasoning (CoT)
**Did step-by-step reasoning improve clarity and usefulness?**

[Your answer]

### 6. Variance Reduction
**Which advanced technique best reduced output variance and improved consistency?**

[Your answer]

### 7. Production Recommendation
**Which technique(s) would you actually use in production? Why?**

[Your answer]

### 8. Technique Combinations
**What combinations of techniques worked especially well together?**

[Your answer]

### 9. Key Learnings
**What did you learn about prompt design that surprised you?**

[Your answer]

### 10. Future Iterations
**If you had more time, how would you improve your system prompt further?**

[Your answer]

---
# ✅ Project Complete!

## Next Steps:
1. Review all generated outputs
2. Complete the comparison matrix with ratings
3. Answer all reflection questions
4. Export this notebook and results.json
5. Compile final submission document

## Files to Submit:
- [ ] This completed notebook
- [ ] App_Concept.md
- [ ] results.json
- [ ] final_system_prompt.txt
- [ ] Reflection document
- [ ] Comparison matrix

Good luck! 🚀
---